In [1]:
import numpy as np
import pandas as pd

TRAIN_LABEL = 'data/labeledTrainData.tsv'
TRAIN_UNLABEL = 'data/unlabeledTrainData.tsv'
TEST = 'data/testData.tsv'

all_data = pd.read_csv(TRAIN_LABEL, header=0, delimiter="\t", quoting=3)
bonus_data = pd.read_csv(TRAIN_UNLABEL, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST, header=0, delimiter="\t", quoting=3)

print(all_data.shape)
print(bonus_data.shape)
print(test.shape)

print(all_data.head(2))

(25000, 3)
(50000, 2)
(25000, 2)
         id  sentiment                                             review
0  "5814_8"          1  "With all this stuff going down at the moment ...
1  "2381_9"          1  "\"The Classic War of the Worlds\" by Timothy ...


In [2]:
import re

def clean_text(text):
    text = text.lower()
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

all_data['review'] = all_data['review'].apply(clean_text)
bonus_data['review'] = bonus_data['review'].apply(clean_text)
test['review'] = test['review'].apply(clean_text)

print(all_data['review'][0])

with all this stuff going down at the moment with mj ive started listening to his music watching the odd documentary here and there watched the wiz and watched moonwalker again maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent moonwalker is part biography part feature film which i remember going to see at the cinema when it was originally released some of it has subtle messages about mjs feeling towards the press and also the obvious message of drugs are bad mkaybr br visually impressive but of course this is all about michael jackson so unless you remotely like mj in anyway then you are going to hate this and find it boring some may call mj an egotist for consenting to the making of this movie but mj and most of his fans would say that he made it for the fans which if true is really nice of himbr br the actual feature film bit when it finally starts is only on for 20 min

In [3]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(all_data, test_size=0.2, random_state=42, shuffle=True)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train['review']).toarray()
y_train = train['sentiment']

X_val = vectorizer.transform(val['review']).toarray()
y_val = val['sentiment']

X_test = vectorizer.transform(test['review']).toarray()

In [5]:
print(X_train)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [6]:
vocab = vectorizer.get_feature_names_out()
print(f'Vocabulary size: {len(vocab)}')

Vocabulary size: 106644


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(random_state=42, n_jobs=-1)

model.fit(X_train, y_train)

val_preds = model.predict(X_val)

auc = roc_auc_score(y_val, val_preds)
print(f'Validation AUC: {auc}')

Validation AUC: 0.8294190272430135


In [ ]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(all_data['review']).toarray()
y_train = all_data['sentiment']
X_test = vectorizer.transform(test['review']).toarray()

model = RandomForestClassifier(random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'sentiment': test_preds
})

submission.to_csv('submissions/tfidf_rf.csv', index=False)